# Root-centered mocap test

Production transform: `utils_mocap_viz/worldpos_transform_v2.py` — horizontal root center (X/Z), Y fixed (ground at 0), frontal yaw via `frontal_method` over `markers`.  
`MocapVisualizerBase` loads `bvh_to_csv_centered/{base}_worldpos_rc_{markers}_{method}.csv` (one cache file per option).

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

from utils_mocap_viz.worldpos_transform_v2 import (
    ensure_root_centered_worldpos_csv,
    transform_worldpos_to_root_centered,
    marker_names_from_columns,
)
from utils_mocap_viz.worldpos_transform_v2 import root_centered_worldpos_path
from utils_mocap_viz.mocap_visualizer_F_view import MocapVisualizerFront
from utils_mocap_viz.mocap_visualizer_LS_view import MocapVisualizerLeftSide
from utils_mocap_viz.mocap_visualizer_RS_view import MocapVisualizerRightSide


with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

c=0
for piece in piece_list:
    piece = "BKO_E2_D3_03_Wasulunka"
    c+=1
    if c>1:
        break
    # --- config ---
    BASE_NAME = f"{piece}_T"
    DIR_CSV = "bvh_to_csv"
    BVH_FILE = os.path.join("data", "bvh_files", f"{BASE_NAME}.bvh")


    SOURCE_CSV = os.path.join(DIR_CSV, f"{BASE_NAME}_worldpos.csv")
    ROOT_CENTERED_CSV = root_centered_worldpos_path(BASE_NAME, DIR_CSV)

    TEST_DIR = "testing_output"
    os.makedirs(TEST_DIR, exist_ok=True)
    OUTPUT_DIR = os.path.join(TEST_DIR, BASE_NAME)

    ROOT = "Hips"
    LEFT_MARKER = "LeftShoulder"
    RIGHT_MARKER = "RightShoulder"
    # FRONTAL_METHOD = "none"  # "frame" | "mean" (JYU mc2frontal)

    START_TIME = 217.0
    END_TIME = 234.0


    # ------------------------------

    raw_df = pd.read_csv(SOURCE_CSV)
    print(f"Loaded {len(raw_df)} frames from {SOURCE_CSV}")

    ROOT_CENTERED_CSV = ensure_root_centered_worldpos_csv(SOURCE_CSV, force=True)
    transformed_df = pd.read_csv(ROOT_CENTERED_CSV)
    print(f"Root-centered CSV -> {ROOT_CENTERED_CSV}")

    # ------------------------------



    VIEW_CONFIG = [
        ("front", MocapVisualizerFront, True),
        # ("left", MocapVisualizerLeftSide, False),
        # ("right", MocapVisualizerRightSide, False),
    ]

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    generated = {}

    for view_name, vis_cls, show_info in VIEW_CONFIG:
        view_dir = os.path.join(OUTPUT_DIR, f"{view_name}_view")
        os.makedirs(view_dir, exist_ok=True)
        output_file = os.path.join(
            view_dir,
            f"{view_name}_view_{START_TIME:.1f}_{END_TIME:.1f}.mp4",
        )
        print(f"\nGenerating {view_name} view -> {output_file}")
        visualizer = vis_cls(BVH_FILE, debug=True)
        visualizer.generate_video(
            output_file=output_file,
            start_time=START_TIME,
            end_time=END_TIME,
            output_fps=24,
            video_size=(1280, 720),
            show_info=show_info,
        )
        visualizer.plotter.close()
        generated[view_name] = output_file

    print("\nGenerated:")
    for view_name, path in generated.items():
        print(f"  {view_name}: {path}")

# Frontal-alignment comparison (A / B / C) — `worldpos_transform_v2`

Root cause of "not front fixed": production bakes **one whole-clip `"mean"` yaw** into the cached CSV, so any segment whose facing differs from the clip average renders rotated (sideways/back).

This compares, on the 5 flagged windows, four strategies using the **hip+shoulder averaged** facing (`markers="both"`):

| | strategy | fixes segment offset? | removes within-window turning? |
|---|---|---|---|
| **NOW** | whole-clip mean (production) | no | no |
| **A** | per-window mean | yes | no |
| **B** | per-frame (JYU `frame`) | yes | yes (always frontal) |
| **C** | per-frame smoothed | yes | yes, minus high-freq jitter |

Two outputs per window: a **facing-error vs time** plot (0° = front) and a **front-projection montage** (what the front camera sees; blue=left, red=right).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils_mocap_viz.worldpos_transform_v2 import (
    marker_names_from_columns,
    stack_markers,
    root_center_horizontal,
    apply_frontal,
    frontal_residual_deg,
)

FPS = 240
MARKERS = "both"      # hip-line + shoulder-line averaged
SMOOTH_SEC = 0.4      # low-pass window for strategy C

SKELETON = [
    'Hips', 'Chest', 'Chest2', 'Chest3', 'Chest4', 'Neck', 'Head',
    'RightCollar', 'RightShoulder', 'RightElbow', 'RightWrist',
    'LeftCollar', 'LeftShoulder', 'LeftElbow', 'LeftWrist',
    'RightHip', 'RightKnee', 'RightAnkle', 'RightToe',
    'LeftHip', 'LeftKnee', 'LeftAnkle', 'LeftToe',
]
CONNECTIONS = [
    ('Hips', 'Chest'), ('Chest', 'Chest2'), ('Chest2', 'Chest3'),
    ('Chest3', 'Chest4'), ('Chest4', 'Neck'), ('Neck', 'Head'),
    ('Chest4', 'RightCollar'), ('RightCollar', 'RightShoulder'),
    ('RightShoulder', 'RightElbow'), ('RightElbow', 'RightWrist'),
    ('Chest4', 'LeftCollar'), ('LeftCollar', 'LeftShoulder'),
    ('LeftShoulder', 'LeftElbow'), ('LeftElbow', 'LeftWrist'),
    ('Hips', 'RightHip'), ('RightHip', 'RightKnee'),
    ('RightKnee', 'RightAnkle'), ('RightAnkle', 'RightToe'),
    ('Hips', 'LeftHip'), ('LeftHip', 'LeftKnee'),
    ('LeftKnee', 'LeftAnkle'), ('LeftAnkle', 'LeftToe'),
]

# (base, t0, t1, note) — the windows you flagged
WINDOWS = [
    ("BKO_E3_D6_01_Maraka_T",   103, 108, "sideways"),
    ("BKO_E2_D3_01_Maraka_T",   210, 215, "rotations"),
    ("BKO_E3_D6_06_Manjanin_T", 100, 105, "not front fixed"),
    ("BKO_E3_D6_03_Wasulunka_T", 176, 182, "sideways"),
    ("BKO_E3_D6_12_Suku_T",     129, 135, "fine (control)"),
]

OUT = "testing_output/frontal_compare"
os.makedirs(OUT, exist_ok=True)


def load_clip(base):
    """Read only skeleton-marker columns from the raw worldpos CSV."""
    keep = set(['Time'] + [f'{m}.{a}' for m in SKELETON for a in 'XYZ'])
    return pd.read_csv(f'bvh_to_csv/{base}_worldpos.csv', usecols=lambda c: c in keep)


def transform_variants(df, t0, t1):
    """Hip-center the whole clip, then build the 4 frontal variants for [t0, t1]."""
    labels = marker_names_from_columns(df.columns)
    pos = stack_markers(df, labels)
    pos = root_center_horizontal(pos, pos[:, labels.index('Hips'), :])

    tmask = (df['Time'].values >= t0) & (df['Time'].values <= t1)
    win = pos[tmask]
    t = df['Time'].values[tmask]

    now_full = apply_frontal(pos, labels, MARKERS, 'mean')  # whole-clip mean, then slice
    variants = {
        'NOW: whole-clip mean': now_full[tmask],
        'A: window mean':       apply_frontal(win, labels, MARKERS, 'mean'),
        'B: per-frame':         apply_frontal(win, labels, MARKERS, 'frame'),
        'C: per-frame smooth':  apply_frontal(win, labels, MARKERS, 'frame_smooth',
                                              fps=FPS, smooth_sec=SMOOTH_SEC),
    }
    return labels, t, variants


def _bone_color(a, b):
    s = a + b
    if 'Left' in s and 'Right' not in s:
        return 'tab:blue'
    if 'Right' in s and 'Left' not in s:
        return 'tab:red'
    return 'black'


def plot_residual(base, t0, t1, note, labels, t, variants):
    fig, ax = plt.subplots(figsize=(10, 3.2))
    for name, arr in variants.items():
        ax.plot(t, frontal_residual_deg(arr, labels, MARKERS), lw=1.2, label=name)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_ylim(-185, 185)
    ax.set_yticks([-180, -90, 0, 90, 180])
    ax.set_xlabel('time (s)')
    ax.set_ylabel('facing error (deg)\n0 = front')
    ax.set_title(f'{base}  [{t0}-{t1}s]  {note}')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    fig.tight_layout()
    p = f'{OUT}/{base}_{t0}_{t1}_residual.png'
    fig.savefig(p, dpi=110)
    plt.show()
    return p


def plot_montage(base, t0, t1, note, labels, t, variants, nframes=4):
    idx = np.linspace(0, len(t) - 1, nframes).astype(int)
    names = list(variants)
    li = {lab: i for i, lab in enumerate(labels)}

    allpts = np.concatenate(list(variants.values()), axis=0)
    L = np.percentile(np.abs(allpts[:, :, 0]), 99) * 1.15
    ymin, ymax = allpts[:, :, 1].min() - 0.1, allpts[:, :, 1].max() + 0.1

    fig, axes = plt.subplots(len(names), nframes,
                             figsize=(2.4 * nframes, 2.7 * len(names)), squeeze=False)
    for r, name in enumerate(names):
        arr = variants[name]
        for c, fi in enumerate(idx):
            ax = axes[r][c]
            P = arr[fi]
            hx = P[li['Hips'], 0]
            for aJ, bJ in CONNECTIONS:
                if aJ in li and bJ in li:
                    ax.plot([P[li[aJ], 0] - hx, P[li[bJ], 0] - hx],
                            [P[li[aJ], 1], P[li[bJ], 1]],
                            color=_bone_color(aJ, bJ), lw=1.6)
            res = frontal_residual_deg(P[None], labels, MARKERS)[0]
            ax.set_xlim(-L, L)
            ax.set_ylim(ymin, ymax)
            ax.set_aspect('equal')
            ax.set_xticks([]); ax.set_yticks([])
            ax.text(0.5, 0.98, f'{res:+.0f}\u00b0', transform=ax.transAxes,
                    ha='center', va='top', fontsize=7, color='dimgray')
            if r == 0:
                ax.set_title(f't={t[fi]:.1f}s', fontsize=8)
            if c == 0:
                ax.set_ylabel(name, fontsize=8)
    fig.suptitle(f'Front projection (X horiz, Y vert) \u2014 {base} [{t0}-{t1}s] {note}\n'
                 f'blue=left  red=right   \u00b1deg = facing error (0 = front)', fontsize=9)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    p = f'{OUT}/{base}_{t0}_{t1}_montage.png'
    fig.savefig(p, dpi=110)
    plt.show()
    return p

print("setup ready")

In [ ]:
def circ_std_deg(deg):
    r = np.radians(deg)
    R = np.hypot(np.mean(np.cos(r)), np.mean(np.sin(r)))
    return np.degrees(np.sqrt(max(0.0, -2 * np.log(max(R, 1e-12)))))


print(f"{'window':34s} {'variant':22s} {'mean':>7s} {'std':>6s} {'|max|':>7s}")
print("-" * 80)
for base, t0, t1, note in WINDOWS:
    df = load_clip(base)
    labels, t, variants = transform_variants(df, t0, t1)
    for name, arr in variants.items():
        res = frontal_residual_deg(arr, labels, MARKERS)
        tag = f"{base} {t0}-{t1}"
        print(f"{tag:34s} {name:22s} {res.mean():7.1f} {circ_std_deg(res):6.1f} "
              f"{np.max(np.abs(res)):7.1f}")
    plot_residual(base, t0, t1, note, labels, t, variants)
    plot_montage(base, t0, t1, note, labels, t, variants)
    print("-" * 80)